# Almacenamiento y Procesamiento de Grandes Datos

## Subtaller 3: Apache Spark — Arquitectura Medallion (Colab)

## Objetivo
Aprender los fundamentos de Apache Spark mediante la implementación de una arquitectura **Medallion** (Bronze → Silver → Gold) usando un dataset de ventas de e-commerce.

## Arquitectura Medallion
```
JSON crudo                 Bronze               Silver                  Gold
(data/)          →   (datos sin limpiar)  →  (datos limpios)  →  (datos agregados)
ventas_ecommerce.json   Parquet raw         Parquet limpio         Parquet analítico
```

## Contenido
1. Instalación y verificación de PySpark
2. Conceptos clave de Spark
3. Crear SparkSession
4. Capa Bronze — Ingestar datos crudos
5. Capa Silver — Limpiar y transformar
6. Capa Gold — Agregar para análisis
7. Consultas analíticas sobre Gold

---
## 1. Instalación y verificación

In [ ]:
# Instalación automática para Colab (solo si hace falta)
import importlib.util
import subprocess
import sys

if importlib.util.find_spec('pyspark') is None:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'pyspark'])

import pyspark
print(f'PySpark version: {pyspark.__version__}')

---
## 2. Conceptos clave de Spark

| Concepto | Descripción |
|---|---|
| **SparkSession** | Punto de entrada a Spark. Una por aplicación. |
| **DataFrame** | Tabla distribuida con columnas tipadas. Equivale a un DataFrame de Pandas pero distribuido. |
| **RDD** | Resilient Distributed Dataset — capa de bajo nivel (hoy se prefiere DataFrame). |
| **Transformación** | Operación lazy: `filter`, `select`, `groupBy`. No ejecuta hasta una acción. |
| **Acción** | Dispara la ejecución: `show()`, `count()`, `write`, `collect()`. |
| **Partición** | Chunk del dataset distribuido entre nodos (en local, entre CPU cores). |
| **Lazy Evaluation** | Spark construye un plan de ejecución y lo optimiza antes de correrlo. |
| **Catalyst Optimizer** | Motor interno que optimiza el plan de consulta automáticamente. |

---
## 3. Crear SparkSession

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *
import os

# SparkSession es el punto de entrada de cualquier aplicación Spark
spark = SparkSession.builder \
    .appName('TallerMedallion') \
    .master('local[*]') \
    .config('spark.sql.shuffle.partitions', '4') \
    .getOrCreate()

# Reducir logs para mayor legibilidad
spark.sparkContext.setLogLevel('WARN')

print(f'Spark version: {spark.version}')
print(f'Master: {spark.sparkContext.master}')

In [ ]:
# Definir rutas de la arquitectura Medallion
BASE_PATH = './medallion'
BRONZE_PATH = f'{BASE_PATH}/bronze/ventas'
SILVER_PATH = f'{BASE_PATH}/silver/ventas'
GOLD_PATH   = f'{BASE_PATH}/gold'

# Ruta del dataset fuente
SOURCE_JSON = './data/ventas_ecommerce.json'

# Crear directorios locales
for path in [BRONZE_PATH, SILVER_PATH, GOLD_PATH]:
    os.makedirs(path, exist_ok=True)

print('Rutas creadas:')
for path in [BRONZE_PATH, SILVER_PATH, GOLD_PATH]:
    print(f'  {path}')

---
## 4. Capa Bronze — Ingesta de datos crudos

> **Bronze**: datos tal como llegan de la fuente. Sin limpieza. Solo se agrega metadata de ingesta.

In [ ]:
# Leer el JSON crudo
df_raw = spark.read \
    .option('multiline', 'true') \
    .json(SOURCE_JSON)

print('=== Schema del dataset crudo ===')
df_raw.printSchema()

In [ ]:
# Explorar los datos
print(f'Total registros: {df_raw.count()}')
df_raw.show(5, truncate=False)

In [ ]:
# Bronze: agregar metadata de ingesta y guardar en Parquet
from datetime import datetime

df_bronze = df_raw \
    .withColumn('ingestion_timestamp', F.current_timestamp()) \
    .withColumn('source_file', F.lit(SOURCE_JSON)) \
    .withColumn('ingestion_date', F.current_date())

# Escribir en formato Parquet (columnar, eficiente para analytics)
df_bronze.write \
    .mode('overwrite') \
    .partitionBy('ingestion_date') \
    .parquet(BRONZE_PATH)

print(f'Bronze guardado en: {BRONZE_PATH}')
print(f'Registros en Bronze: {df_bronze.count()}')

In [ ]:
# Verificar lo que se escribió en Bronze
df_bronze_check = spark.read.parquet(BRONZE_PATH)
df_bronze_check.select(
    'order_id', 'customer_name', 'product_name',
    'unit_price', 'status', 'ingestion_timestamp'
).show(5, truncate=False)

---
## 5. Capa Silver — Limpieza y transformaciones

> **Silver**: datos limpios, validados y enriquecidos. Se corrigen tipos, se eliminan duplicados y se calculan columnas derivadas.

In [ ]:
# Leer desde Bronze
df_b = spark.read.parquet(BRONZE_PATH)

# 1. Verificar nulos
print('=== Conteo de nulos por columna ===')
df_b.select([
    F.count(F.when(F.col(c).isNull(), c)).alias(c)
    for c in ['order_id', 'customer_id', 'product_id', 'quantity', 'unit_price', 'status']
]).show()

In [ ]:
# 2. Verificar duplicados
total = df_b.count()
distinct = df_b.dropDuplicates(['order_id']).count()
print(f'Total registros: {total}')
print(f'Registros únicos por order_id: {distinct}')
print(f'Duplicados: {total - distinct}')

In [ ]:
# 3. Verificar valores de la columna status
df_b.groupBy('status').count().orderBy('count', ascending=False).show()

In [ ]:
# Transformaciones Silver
df_silver = df_b \
    .dropDuplicates(['order_id']) \
    .filter(F.col('order_id').isNotNull()) \
    .filter(F.col('customer_id').isNotNull()) \
    .withColumn('order_date', F.to_date('order_date', 'yyyy-MM-dd')) \
    .withColumn('order_year',  F.year('order_date')) \
    .withColumn('order_month', F.month('order_date')) \
    .withColumn('unit_price',  F.col('unit_price').cast(DoubleType())) \
    .withColumn('quantity',    F.col('quantity').cast(IntegerType())) \
    .withColumn('discount',    F.col('discount').cast(DoubleType())) \
    .withColumn('gross_amount', F.round(F.col('quantity') * F.col('unit_price'), 2)) \
    .withColumn('net_amount',   F.round(
        F.col('quantity') * F.col('unit_price') * (1 - F.col('discount')), 2
    )) \
    .withColumn('is_cancelled', F.when(F.col('status') == 'cancelado', True).otherwise(False)) \
    .withColumn('status', F.lower(F.trim('status'))) \
    .drop('ingestion_date', 'source_file')

print(f'Silver: {df_silver.count()} registros')
df_silver.printSchema()

In [ ]:
# Guardar Silver particionado por año y mes
df_silver.write \
    .mode('overwrite') \
    .partitionBy('order_year', 'order_month') \
    .parquet(SILVER_PATH)

print(f'Silver guardado en: {SILVER_PATH}')

In [ ]:
# Vista previa de Silver
spark.read.parquet(SILVER_PATH).select(
    'order_id', 'customer_name', 'product_name', 'category',
    'gross_amount', 'net_amount', 'status', 'order_date'
).show(8, truncate=False)

---
## 6. Capa Gold — Agregaciones para análisis

> **Gold**: datos listos para consumo analítico. Se crean tablas de hechos y dimensiones, KPIs y reportes.

In [ ]:
df_s = spark.read.parquet(SILVER_PATH)
df_active = df_s.filter(F.col('status') != 'cancelado')

# --- Gold 1: KPIs por mes ---
gold_monthly = df_active \
    .groupBy('order_year', 'order_month') \
    .agg(
        F.count('order_id').alias('total_orders'),
        F.countDistinct('customer_id').alias('unique_customers'),
        F.round(F.sum('gross_amount'), 2).alias('gross_revenue'),
        F.round(F.sum('net_amount'), 2).alias('net_revenue'),
        F.round(F.avg('net_amount'), 2).alias('avg_order_value')
    ) \
    .orderBy('order_year', 'order_month')

gold_monthly.write.mode('overwrite').parquet(f'{GOLD_PATH}/kpis_mensuales')
print('Gold: KPIs mensuales')
gold_monthly.show()

In [ ]:
# --- Gold 2: Ventas por categoría de producto ---
gold_category = df_active \
    .groupBy('category') \
    .agg(
        F.count('order_id').alias('num_orders'),
        F.sum('quantity').alias('units_sold'),
        F.round(F.sum('net_amount'), 2).alias('net_revenue'),
        F.round(F.avg('unit_price'), 2).alias('avg_price')
    ) \
    .orderBy('net_revenue', ascending=False)

gold_category.write.mode('overwrite').parquet(f'{GOLD_PATH}/ventas_por_categoria')
print('Gold: Ventas por categoría')
gold_category.show()

In [ ]:
# --- Gold 3: Top clientes por revenue ---
gold_customers = df_active \
    .groupBy('customer_id', 'customer_name', 'city') \
    .agg(
        F.count('order_id').alias('num_orders'),
        F.round(F.sum('net_amount'), 2).alias('total_spent'),
        F.round(F.avg('net_amount'), 2).alias('avg_order')
    ) \
    .orderBy('total_spent', ascending=False)

gold_customers.write.mode('overwrite').parquet(f'{GOLD_PATH}/top_clientes')
print('Gold: Top clientes')
gold_customers.show(10)

In [ ]:
# --- Gold 4: Top productos ---
gold_products = df_active \
    .groupBy('product_id', 'product_name', 'category') \
    .agg(
        F.sum('quantity').alias('units_sold'),
        F.round(F.sum('net_amount'), 2).alias('net_revenue')
    ) \
    .orderBy('net_revenue', ascending=False)

gold_products.write.mode('overwrite').parquet(f'{GOLD_PATH}/top_productos')
print('Gold: Top productos')
gold_products.show()

---
## 7. Consultas analíticas sobre Gold

In [ ]:
# Registrar Silver como vista temporal para usar Spark SQL
df_s.createOrReplaceTempView('ventas')

# Spark SQL — exactamente igual a SQL estándar
spark.sql('''
SELECT
    category,
    COUNT(order_id)       AS num_ordenes,
    SUM(quantity)         AS unidades,
    ROUND(SUM(net_amount),2) AS ingresos_netos
FROM ventas
WHERE status != 'cancelado'
GROUP BY category
ORDER BY ingresos_netos DESC
''').show()

In [ ]:
# Tasa de cancelación
spark.sql('''
SELECT
    CONCAT(order_year, '-', LPAD(order_month, 2, '0')) AS mes,
    COUNT(*) AS total,
    SUM(CASE WHEN status = 'cancelado' THEN 1 ELSE 0 END) AS cancelados,
    ROUND(100.0 * SUM(CASE WHEN status = 'cancelado' THEN 1 ELSE 0 END) / COUNT(*), 1) AS pct_cancelacion
FROM ventas
GROUP BY mes
ORDER BY mes
''').show()

In [ ]:
# Window function — ranking de clientes por revenue dentro de cada ciudad
from pyspark.sql.window import Window

window_city = Window.partitionBy('city').orderBy(F.desc('total_spent'))

spark.read.parquet(f'{GOLD_PATH}/top_clientes') \
    .withColumn('rank_in_city', F.rank().over(window_city)) \
    .filter(F.col('rank_in_city') == 1) \
    .select('city', 'customer_name', 'total_spent', 'num_orders') \
    .orderBy('total_spent', ascending=False) \
    .show()

In [ ]:
# Cerrar SparkSession
spark.stop()
print('SparkSession cerrada.')